# 🎬 Pipeline Multilíngue — Versão Refatorada

**Arquitetura modular** com checkpoint por fase, legendas ASS (10× mais rápido que drawtext) e cache de classificação morfológica.

| Célula | Fase | Descrição |
|--------|------|-----------|
| 0 | Setup | Instalar deps, montar Drive, copiar módulos |
| B0 | Opcional | Baixar legendas YouTube (EN/ES/FR) via yt-dlp |
| Init | — | Instanciar config, groq e pipeline |
| 1–8 | Fases | Executar cada fase individualmente |
| RUN | Run All | Executa tudo com checkpoint automático |
| Utils | — | Ferramentas auxiliares |

> **Para retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`
>
> **Fases válidas:** `audio_gerado` | `srt_pt_bruto` | `srt_pt_corrigido` | `srt_traduzidos` | `classificacoes_feitas` | `clipes_cortados` | `video_base_criado` | `legendas_queimadas`

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup                                          ║
# ╚══════════════════════════════════════════════════════════════╝

# Dependências de sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Dependências Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp
print('✅ pacotes Python')

# Montar Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
print('✅ Drive montado')

# Copiar módulos do Drive → /content/pipeline
import shutil, os, sys
PASTA_MODULOS = '/content/drive/MyDrive/pipeline'   # <-- AJUSTE se necessário
DESTINO       = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')
else:
    print('⚠️  PASTA_MODULOS não encontrada. Ajuste o caminho acima.')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('✅ Pronto.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — OPCIONAL: legendas YouTube (EN/ES/FR)         ║
# ║  Execute antes das fases 1–8 para traduções mais fiéis.   ║
# ║  Se pular, o Groq traduz a partir do texto PT.            ║
# ╚══════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
from drive_utils import DriveClient
from config import PipelineConfig

cfg   = PipelineConfig()
drive = DriveClient.get()

# Baixar cookies (necessário para o YouTube)
drive.download_se_ausente(cfg.ID_PASTA_COOKIES, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

# URLs dos vídeos com legendas oficiais — preencha abaixo
URLS = {
    'en': 'https://www.youtube.com/watch?v=COLOQUE_URL_EN',
    'es': 'https://www.youtube.com/watch?v=COLOQUE_URL_ES',
    'fr': 'https://www.youtube.com/watch?v=COLOQUE_URL_FR',
}

for lang, url in URLS.items():
    if 'COLOQUE_URL' in url:
        print(f'⏭️  {lang.upper()}: URL não configurada')
        continue
    nome_srt = cfg.nome_srt(lang)
    print(f'⬇️  Baixando {lang.upper()}...')
    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang,
        '--write-auto-sub', '--skip-download',
        '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}',
        *cookies_flag, url
    ]
    subprocess.run(cmd, capture_output=True)
    # renomear se yt-dlp adicionou sufixo extra
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt}')
        break
    else:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')

print('B0 concluído.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  INICIALIZAÇÃO — Rodar uma vez após o Setup                ║
# ╚══════════════════════════════════════════════════════════════╝

from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# ── Configuração (edite para trocar a oração) ────────────────────────────────
config = PipelineConfig(
    NOME_ORACAO  = 'pai_nosso',
    # Para outra oração, descomente e ajuste:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',
)

groq     = GroqClient(api_key=userdata.get('GROQ_KEY'))
pipeline = VideoPipeline(config, groq)

# Estado atual
cp = Checkpoint()
print(cp.resumo())
print(f'\n▶  Próxima fase: {cp.proxima_fase_pendente() or "(tudo concluído)"}')

In [ ]:
# ── FASE 1: Gerar áudio com Edge TTS ────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')

In [ ]:
# ── FASE 2: Transcrever com Whisper ─────────────────────────────────────────
srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
for leg in ler_srt(srt_bruto)[:5]:
    print(f'  [{leg.inicio_str}]  {leg.texto}')

In [ ]:
# ── FASE 3: Corrigir PT com Groq + eliminar gaps ────────────────────────────
legendas_pt = pipeline.fase3_corrigir_pt()

print(f'{len(legendas_pt)} legendas PT:')
for leg in legendas_pt:
    print(f'  #{leg.id:02d}  {leg.texto}')

In [ ]:
# ── FASE 4: Traduzir EN/ES/FR ────────────────────────────────────────────────
from srt_utils import ler_srt
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)

legendas_idiomas = pipeline.fase4_traduzir(legendas_pt)

# Preview comparativo (primeiras 2 legendas)
for i in range(min(2, len(legendas_pt))):
    for lang in config.IDIOMAS:
        if lang in legendas_idiomas:
            print(f'  {lang.upper()}: {legendas_idiomas[lang][i].texto}')
    print()

In [ ]:
# ── FASE 5: Classificação morfológica (com cache JSON) ──────────────────────
# Cache local: se o JSON já existe e é válido, não chama o Groq.
# Para forçar: pipeline._clf.invalidar_cache('pt')

from srt_utils import ler_srt
if 'legendas_idiomas' not in dir() or not legendas_idiomas:
    legendas_pt = ler_srt(config.NOME_SRT_PT)
    legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)

legendas_idiomas = pipeline.fase5_classificar(legendas_idiomas)

# Preview
leg = legendas_idiomas['pt'][0]
print(f'PT — "{leg.texto}"')
for p in leg.palavras:
    print(f'  {p.texto:<20} {p.classe}')

In [ ]:
# ── FASE 6: Baixar e cortar clipes ──────────────────────────────────────────
from srt_utils import ler_srt
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)

clipes = pipeline.fase6_baixar_clipes(legendas_pt)
print(f'{len(clipes)} clipes prontos')

In [ ]:
# ── FASE 7: Vídeo base (crédito + narração + trilha) ────────────────────────
if 'clipes' not in dir() or not clipes:
    clipes = pipeline._clipes_do_checkpoint()

video_base = pipeline.fase7_criar_video_base(clipes)
print(f'✅ {video_base}  ({video_base.stat().st_size/1_048_576:.1f} MB)')

In [ ]:
# ── FASE 8: Queimar legendas ASS → vídeo final ──────────────────────────────
# Um único arquivo .ass com 4 idiomas → 1 passe de render (10× mais rápido)

from srt_utils import ler_srt
if 'legendas_idiomas' not in dir() or not legendas_idiomas:
    legendas_pt = ler_srt(config.NOME_SRT_PT)
    legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)
    legendas_idiomas = pipeline._carregar_classificacoes(legendas_idiomas)

video_final = pipeline.fase8_queimar_legendas(legendas_idiomas)
print(f'🎉 {video_final}  ({video_final.stat().st_size/1_048_576:.1f} MB)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  RUN ALL — Pipeline completo com checkpoint                ║
# ╚══════════════════════════════════════════════════════════════╝

# Opções de uso:
#   pipeline.run()                              # executa do zero / continua
#   pipeline.run(from_phase='clipes_cortados')  # reinicia a partir de uma fase

video_final = pipeline.run(
    # from_phase='clipes_cortados'
)

print(f'\n🎬 Vídeo final: {video_final}')
print(pipeline._cp.resumo())

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILITÁRIOS                                               ║
# ╚══════════════════════════════════════════════════════════════╝
from checkpoint import Checkpoint
from pathlib import Path

cp = Checkpoint()
print(cp.resumo())

# ── Ações rápidas (descomente a que precisar) ──────────────────────────────

# Resetar tudo:
# cp.resetar_tudo()

# Reiniciar a partir de uma fase:
# cp.reiniciar_de('classificacoes_feitas')

# Forçar reclassificação de um idioma:
# pipeline._clf.invalidar_cache('en')

# Ver ASS gerado:
# ass = Path(f'legendas_{config.NOME_ORACAO}.ass')
# print(ass.read_text()[:2000])

# Limpar temporários:
# pipeline.limpar_temporarios()

# Inspecionar SRT:
# from srt_utils import ler_srt
# for leg in ler_srt(config.nome_srt('fr')):
#     print(f'[{leg.inicio_str}]  {leg.texto}')